# ch02 — Gen1 statistics: PCA-T²/SPE

Applied to SMD (needs local data/) or to synthetic multivariate data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
# Use SMD if available, otherwise synthetic multivariate data
try:
    from tsad_forge.data.registry import load_dataset
    ds = load_dataset("smd", machine="machine-1-1")
    print("using SMD machine-1-1")
except FileNotFoundError:
    ds = generate_synthetic(n_dims=8, n_events=6, seed=2)
    print("using synthetic (run `tsad-forge download smd` for real data)")

In [ ]:
from tsad_forge.models.registry import get_model
from tsad_forge.evaluation.protocol import zscore_normalize
train, test = zscore_normalize(ds.train, ds.test)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, mode in zip(axes, ["t2", "spe", "combined"]):
    scores = get_model("pca_t2spe", mode=mode).fit(train).score(test)
    ax.plot(scores, lw=0.6)
    ax.fill_between(np.arange(len(scores)), *ax.get_ylim(),
                    where=ds.labels.astype(bool), alpha=0.25, color="red")
    ax.set_title(f"PCA-{mode.upper()} — T² (PC subspace) and SPE (residual subspace) catch different anomalies")
plt.tight_layout()